# Zomato Restaurant & Review Data Analysis
**Prepared by:** Reenotha S.  
**Project type:** Data Analytics | **Tools:** Python, Pandas, Matplotlib, Seaborn, Google Colab

## Goal
Analyze restaurant listings and ratings to identify rating patterns, popular cuisines, location hotspots, price-rating relationships, and practical recommendations for a food-discovery platform.

## How to run
1. Open this notebook in Google Colab.
2. Run cells from top to bottom.
3. When asked, upload the Zomato CSV dataset downloaded from the dataset source shown in the assignment.
4. The notebook cleans the data, creates visualizations, prints key findings, and exports a cleaned CSV.
5. Use the generated findings in the accompanying PDF report; update the report's numeric findings after running on the exact dataset.

In [ ]:
# Install/import libraries
import os, re, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
print("Libraries loaded successfully.")

## 1. Load the dataset
Upload the CSV file used for this assignment. The code will also try to locate a CSV already present in the Colab session.

In [ ]:
csv_candidates = glob.glob("/content/*.csv") + glob.glob("/content/**/*.csv", recursive=True)
csv_candidates = list(dict.fromkeys(csv_candidates))
if csv_candidates:
    print("CSV files found:", csv_candidates[:10])
    choice = input("Type the full path of the dataset CSV, or press Enter to upload a file: ").strip()
else:
    choice = ""

if choice and os.path.exists(choice):
    raw = pd.read_csv(choice, encoding_errors="replace")
else:
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_names:
        raise ValueError("Please upload a CSV dataset.")
    raw = pd.read_csv(csv_names[0], encoding_errors="replace")

print("Original dataset shape:", raw.shape)
display(raw.head())
print("Columns:", raw.columns.tolist())

## 2. Standardize columns and inspect data
Column names are normalized so the notebook can work with common versions of the Zomato dataset.

In [ ]:
def normalize_col(c):
    c = str(c).strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c).strip("_")
    return c

df = raw.copy()
df.columns = [normalize_col(c) for c in df.columns]
print("Normalized columns:", df.columns.tolist())
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing_values": df.isna().sum()}))
print("Duplicate rows:", int(df.duplicated().sum()))

## 3. Map common dataset column names
This section maps alternative column names from commonly used Zomato datasets to a standard set of names.

In [ ]:
def find_col(options):
    for opt in options:
        if opt in df.columns:
            return opt
    # fuzzy contains matching
    for c in df.columns:
        for opt in options:
            if opt in c:
                return c
    return None

mapping_options = {
    "restaurant_name": ["restaurant_name", "name", "restaurant"],
    "city": ["city"],
    "locality": ["locality", "area", "location"],
    "cuisines": ["cuisines", "cuisine"],
    "cost_for_two": ["average_cost_for_two", "approx_cost_for_two_people", "avg_cost_two_people", "cost_for_two", "cost"],
    "rating": ["aggregate_rating", "rate", "rating", "rate_out_of_5"],
    "votes": ["votes", "num_of_ratings", "rating_votes"],
    "online_delivery": ["has_online_delivery", "online_order", "online_delivery"],
    "table_booking": ["has_table_booking", "book_table", "table_booking"],
    "price_range": ["price_range", "price_segment"]
}
colmap = {standard: find_col(options) for standard, options in mapping_options.items()}
print("Detected column mapping:")
display(pd.DataFrame({"standard_name": list(colmap.keys()), "source_column": list(colmap.values())}))

In [ ]:
# Create standardized working columns without discarding the source columns
for standard, source in colmap.items():
    if source is not None and source != standard:
        df[standard] = df[source]
    elif source is None and standard not in df.columns:
        df[standard] = np.nan

# Clean text fields
for c in ["restaurant_name", "city", "locality", "cuisines", "online_delivery", "table_booking"]:
    df[c] = df[c].astype("string").str.strip()
    df[c] = df[c].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "null": pd.NA})

# Convert rating values such as '4.1/5' to numeric
df["rating"] = pd.to_numeric(
    df["rating"].astype("string").str.extract(r"(\d+(?:\.\d+)?)", expand=False),
    errors="coerce"
)
df.loc[~df["rating"].between(0, 5), "rating"] = np.nan

# Clean cost values such as '1,200' or '₹ 1,200'
df["cost_for_two"] = pd.to_numeric(
    df["cost_for_two"].astype("string").str.replace(r"[^0-9.]", "", regex=True),
    errors="coerce"
)
df["votes"] = pd.to_numeric(df["votes"].astype("string").str.replace(r"[^0-9]", "", regex=True), errors="coerce")
df["price_range"] = pd.to_numeric(df["price_range"], errors="coerce")

# Remove duplicate records and normalize category text
df = df.drop_duplicates().copy()
for c in ["city", "locality", "cuisines"]:
    df[c] = df[c].str.replace(r"\s+", " ", regex=True).str.strip()

# Primary cuisine is the first cuisine listed
df["primary_cuisine"] = df["cuisines"].str.split(",").str[0].str.strip().str.title()
df["rating_status"] = np.where(df["rating"].notna(), "Rated", "Not rated")

# Fill missing categorical values with a transparent label; do not invent ratings
for c in ["city", "locality", "cuisines", "primary_cuisine"]:
    df[c] = df[c].fillna("Unknown")

print("Cleaned dataset shape:", df.shape)
print("Remaining duplicates:", int(df.duplicated().sum()))
display(df[["restaurant_name","city","locality","cuisines","cost_for_two","rating","votes","primary_cuisine"]].head())

## 4. Key performance indicators (KPIs)
All numbers below are calculated from the uploaded dataset at run time.

In [ ]:
total = len(df)
rated = int(df["rating"].notna().sum())
kpis = pd.DataFrame({
    "Metric": ["Restaurant records", "Rated restaurants", "Not-rated restaurants",
               "Average rating (rated only)", "Median cost for two", "Total votes"],
    "Value": [total, rated, total-rated,
              round(df["rating"].mean(), 2) if rated else "Not available",
              round(df["cost_for_two"].median(), 2) if df["cost_for_two"].notna().any() else "Not available",
              int(df["votes"].fillna(0).sum())]
})
display(kpis)

## 5. Visualizations
The notebook generates rating distribution, top cuisines, location hotspots, and price-versus-rating plots.

In [ ]:
# Rating distribution
if df["rating"].notna().any():
    plt.figure(figsize=(8,4))
    sns.histplot(df["rating"].dropna(), bins=15, kde=True)
    plt.title("Distribution of Restaurant Ratings")
    plt.xlabel("Rating out of 5"); plt.ylabel("Restaurant count")
    plt.tight_layout(); plt.show()
else:
    print("No usable rating values were detected.")

In [ ]:
# Most common primary cuisines
cuisine_counts = df.loc[df["primary_cuisine"] != "Unknown", "primary_cuisine"].value_counts().head(10)
if not cuisine_counts.empty:
    plt.figure(figsize=(10,5))
    sns.barplot(x=cuisine_counts.values, y=cuisine_counts.index)
    plt.title("Top 10 Primary Cuisines")
    plt.xlabel("Number of restaurants"); plt.ylabel("Cuisine")
    plt.tight_layout(); plt.show()
display(cuisine_counts.rename_axis("Cuisine").reset_index(name="Restaurant_count"))

In [ ]:
# Top locations by restaurant count
location_col = "locality" if df["locality"].nunique() > 1 else "city"
loc_counts = df.loc[df[location_col] != "Unknown", location_col].value_counts().head(10)
if not loc_counts.empty:
    plt.figure(figsize=(10,5))
    sns.barplot(x=loc_counts.values, y=loc_counts.index)
    plt.title(f"Top 10 Restaurant Hotspots by {location_col.title()}")
    plt.xlabel("Number of restaurants"); plt.ylabel(location_col.title())
    plt.tight_layout(); plt.show()
display(loc_counts.rename_axis(location_col.title()).reset_index(name="Restaurant_count"))

In [ ]:
# Price vs rating relationship
price_rating = df[["cost_for_two", "rating"]].dropna()
if len(price_rating) >= 3:
    plt.figure(figsize=(8,5))
    sns.regplot(data=price_rating, x="cost_for_two", y="rating",
                scatter_kws={"alpha":0.35}, line_kws={"color":"black"})
    plt.title("Cost for Two vs Restaurant Rating")
    plt.xlabel("Cost for two"); plt.ylabel("Rating out of 5")
    plt.tight_layout(); plt.show()
    print("Correlation (cost vs rating):", round(price_rating.corr().iloc[0,1], 3))
else:
    print("Not enough complete cost/rating rows for this comparison.")

In [ ]:
# Cuisine rating comparison (only cuisines with enough observations)
cuisine_rating = (df[df["rating"].notna() & (df["primary_cuisine"] != "Unknown")]
                  .groupby("primary_cuisine")
                  .agg(restaurant_count=("rating","size"), average_rating=("rating","mean"))
                  .query("restaurant_count >= 3")
                  .sort_values("average_rating", ascending=False)
                  .head(10))
if not cuisine_rating.empty:
    cuisine_rating["average_rating"] = cuisine_rating["average_rating"].round(2)
    display(cuisine_rating)
    plt.figure(figsize=(10,5))
    sns.barplot(data=cuisine_rating.reset_index(), x="average_rating", y="primary_cuisine")
    plt.title("Top Cuisines by Average Rating (minimum 3 restaurants)")
    plt.xlabel("Average rating"); plt.ylabel("Primary cuisine")
    plt.tight_layout(); plt.show()

In [ ]:
# Optional heatmap: average rating by top cities/localities
if df["rating"].notna().any():
    top_places = df[location_col].value_counts().head(8).index
    heat = (df[df[location_col].isin(top_places)]
            .groupby(location_col)["rating"].mean().to_frame("Average rating"))
    if not heat.empty:
        plt.figure(figsize=(6, max(3, len(heat)*0.45)))
        sns.heatmap(heat, annot=True, fmt=".2f", cmap="YlGnBu", cbar=True)
        plt.title(f"Average Rating Heatmap by {location_col.title()}")
        plt.tight_layout(); plt.show()

## 6. Automatically generated key findings
These are calculated from the actual uploaded dataset, so they can be used in the final report after execution.

In [ ]:
findings = []
findings.append(f"The dataset contains {len(df):,} restaurant records after duplicate removal.")
if df["rating"].notna().any():
    findings.append(f"The average rating among rated restaurants is {df['rating'].mean():.2f}/5.")
    findings.append(f"{df['rating'].isna().sum():,} records have missing or unusable ratings.")
if df["primary_cuisine"].ne("Unknown").any():
    top_cuisine = df.loc[df["primary_cuisine"]!="Unknown","primary_cuisine"].value_counts().idxmax()
    top_cuisine_n = int(df.loc[df["primary_cuisine"]! = "Unknown","primary_cuisine"].value_counts().max())
    findings.append(f"The most frequently listed primary cuisine is {top_cuisine} ({top_cuisine_n:,} restaurants).")
if df[location_col].ne("Unknown").any():
    top_loc = df.loc[df[location_col]!="Unknown",location_col].value_counts().idxmax()
    top_loc_n = int(df.loc[df[location_col]! = "Unknown",location_col].value_counts().max())
    findings.append(f"The largest hotspot by {location_col} is {top_loc} ({top_loc_n:,} restaurants).")
if df[["cost_for_two","rating"]].dropna().shape[0] >= 3:
    corr = df[["cost_for_two","rating"]].dropna().corr().iloc[0,1]
    strength = "positive" if corr > 0.1 else "negative" if corr < -0.1 else "weak/near-zero"
    findings.append(f"Cost and rating show a {strength} linear relationship (correlation {corr:.2f}).")
for item in findings:
    print("•", item)

## 7. Recommendations for an Alfido Tech-style platform
1. **Cuisine-led discovery:** feature popular cuisines and provide filters for dietary preference, budget, and rating.
2. **Locality-based collections:** create “popular near you” collections for locations with many restaurant listings.
3. **Rating confidence:** show both rating and number of votes; avoid promoting a high rating with very few votes as definitive.
4. **Value-for-money labels:** compare cost for two with ratings within similar price bands instead of assuming expensive restaurants are better.
5. **Restaurant partner insights:** share a simple monthly summary of rating trend, cuisine competitors, and customer engagement with partner restaurants.

## 8. Export cleaned data

In [ ]:
# Save the cleaned data and summary tables so they can be downloaded and included in the submission.
cleaned_path = "/content/zomato_cleaned_data.csv"
df.to_csv(cleaned_path, index=False)
print("Saved:", cleaned_path)
files.download(cleaned_path)

# Export findings text
findings_path = "/content/zomato_key_findings.txt"
with open(findings_path, "w", encoding="utf-8") as f:
    f.write("Zomato Restaurant & Review Data Analysis - Key Findings\n\n")
    f.write("\n".join("- " + x for x in findings))
    f.write("\n\nRecommendations:\n")
    f.write("1. Cuisine-led discovery and useful filters.\n")
    f.write("2. Locality-based restaurant collections.\n")
    f.write("3. Display rating together with vote count.\n")
    f.write("4. Compare value within similar price bands.\n")
    f.write("5. Share partner insights on ratings and engagement.\n")
files.download(findings_path)

## Conclusion
This analysis turns restaurant listing data into practical insights about ratings, cuisines, location demand, and price. The cleaned CSV, charts, and run-time findings can be used in the final internship submission. Make sure to run the notebook with the official dataset supplied for the task before submitting numeric results.